# Trace Engineering for LLM Reasoning + Fine-Tuning Method

**Competition:** NVIDIA Nemotron Model Reasoning Challenge  
**Result:** Kaggle **Gold Medal**, rank **#14 / 4,182 teams**  
**Primary award category:** Best Data / Synthetic Data Method  
**Secondary award category:** Best Fine-tuning Method

---

## Summary

The main idea behind my solution was to write traces that teach the model a **methodology** for each problem type, not just the final answer.

My traces do four things:

1. identify the problem subtype,
2. state the method or search order,
3. execute the method step by step with no skipped reasoning,
4. include fallback behavior when the model is likely to make a local reasoning or verification mistake.

The fallback mechanism was especially important for **Text Cipher**. To push this type to **100% in type-diagnostic training**, I added verification traces with double confirmation for PASS/FAIL decisions. If the confirmation disagrees with the first attempt, the trace explicitly falls back, re-reads the source word, and corrects the intermediate prediction.

The final dataset has **19,404 rows**, trained with a two-epoch approach:

- **Epoch 1:** train on the full **19,404-row** corpus.
- **Epoch 2:** continue fine-tuning with a smaller learning rate on **13,298 rows**: all **9,419 train-origin rows** plus **3,879 synthetic Numeric Equation rows**.

The key training choices were **selective token weighting** and **category-balanced accumulation**.

- **Selective token weighting:** difficult decision tokens receive higher loss weight through `--decision-weight 2`, instead of weighting every token equally.
- **Category-balanced accumulation:** I trained on a single RTX 6000-class GPU with per-device batch size **1** and gradient accumulation **8**. The trainer uses a balanced sampler so each gradient-accumulation window is spread across question categories.

**Result**: Under a tight time and compute budget, I was only able to train **7 full models** in total, all in Google Colab. Given that the internal type-diagnostic ceiling was around **0.89**, I was happy to land at **0.87** and finish with a gold medal.

*Replication.* Clone my GitHub repo [NVIDIA-Nemotron-Model-Reasoning-Challenge](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge)and run 2 commands to train

* python3 train_sft_single_phase.py --decision-weight 2
* python3 train_sft_single_phase_last_dance.py --adapter-dir outputs/sft_single_phase/adapter --decision-weight 2 

I'll conclude this summary with the solve rates. The solve rate percentages are on the real traces.

| Problem type | Notebook section name | Real rows | Synthetic rows | Solved rows | Total rows | Solve rate |
|---|---|---:|---:|---:|---:|---:|
| Gravity | Gravity + Unit Conversion | 1,597 | 0 | 1,597 | 1,597 | **100%** |
| Unit Conversion | Gravity + Unit Conversion | 1,594 | 0 | 1,594 | 1,594 | **100%** |
| Numeral System | Numeral System | 1,576 | 0 | 1,576 | 1,576 | **100%** |
| Text Cipher | Text Cipher | 1,576 | 1,050 | 1,576 | 2,626 | **100%** |
| Bit Manipulation | Bit Manipulation | 1,602 | 4,515 | ~1,410 | 6,117 | **88%** |
| Numeric Transformation Rules | Numeric Equation | 732 | 3,879 | 651 | 4,530 | **89%** |
| Symbol Transformation Rules | Cryptarithm | 823 | 541 | 72 | 1,364 | **8.7%** |
| **Total** |  | **9,500** | **9,985** | **~8,452** | **19,404** | **~89%** |


# 1. Trace Engineering

---

## 1.1 Gravity and Unit Conversion

These two problem types use almost the same trace methodology and flow:

- identify the problem type,
- estimate the hidden numeric factor from all examples,
- apply that factor to the query.

The key point in both problem types is finding the **unknown linear factor** that maps input to output. I use all examples to estimate this factor because if one example has small rounding or computation noise, the aggregate ratio makes the estimate more stable.

The two formulas are below. A key point I noticed when writing traces for these cases is that the model is prone to making mistakes if the computation is not shown step by step, even for simple arithmetic.

```text
Unit Conversion:
output = factor * input
factor = sum(outputs) / sum(inputs)

Gravity:
d = k * t^2
k = sum(distance) / sum(t^2)
```

For example, when summing three numbers `number_1 + number_2 + number_3`, the trace first computes the partial sum:

```text
number_1 + number_2 = partial_sum
partial_sum + number_3 = total_sum
```

<details>
<summary>Unit Conversion example trace</summary>

```text
We find a conversion rule that maps the inputs to outputs by estimating the linear factor from the examples.
Use factor = sum(outputs) / sum(inputs)

Example pairs
5.41 -> 5.29
8.62 -> 8.44
23.19 -> 22.69

Compute sum(inputs)
sum(inputs) = 5.41 + 8.62 + 23.19
5.41 + 8.62 = 14.03
14.03 + 23.19 = 37.22
sum(inputs) = 37.22

Compute sum(outputs)
sum(outputs) = 5.29 + 8.44 + 22.69
5.29 + 8.44 = 13.73
13.73 + 22.69 = 36.42
sum(outputs) = 36.42

Compute factor
factor = sum(outputs) / sum(inputs)
factor = 36.42 / 37.22
factor = 0.9785 to four decimal places

Compute output
Converting 6.31
6.31 * 0.9785 = 6.1743 to four decimal places

Rounding to two decimals gives 6.17

Answer: \boxed{6.17}
```

</details>





## 1.2 Numeral System

For this problem type, we convert the query number using a greedy decomposition. As with Gravity and Unit Conversion, I avoid skipped arithmetic. Even for a simple number like `38`, the trace does not jump directly to `XXXVIII`; it teaches the conversion procedure by writing each repeated subtraction.

The key update is:

```text
current number >= symbol value -> append symbol 
remainder = current number - symbol value
```

For example:

```text
38 >= 10 -> X, remainder 28
28 >= 10 -> X, remainder 18
18 >= 10 -> X, remainder 8
```

<details>
<summary>Numeral System example trace</summary>

```text
We convert the Arabic number to Roman numerals using the standard greedy table.

Converting 38
38 >= 10 -> X, remainder 28
28 >= 10 -> X, remainder 18
18 >= 10 -> X, remainder 8
8 >= 5 -> V, remainder 3
3 >= 1 -> I, remainder 2
2 >= 1 -> I, remainder 1
1 >= 1 -> I, remainder 0

Putting all together, we get XXXVIII

Answer: \boxed{XXXVIII}
```

</details>





## 1.3 Text Cipher

Text Cipher is the problem type where the fallback mechanism became most important. The base methodology is straightforward:

- process the example phrases to build character mappings,
- decode the target from left to right,
- if a word is not fully mapped, scan the 77-word vocabulary for matching patterns,
- verify candidate words against the current mapping,
- add new mappings only after a confirmed PASS.

The failure mode I observed was not that the model completely missed the cipher rule. More often, it had the right mapping but made a small local mistake in a long trace: copying the source word incorrectly, skipping a repeated letter, or accepting/rejecting a candidate too early.

So I added a small amount of targeted data whose purpose was not to teach the cipher itself, but to teach **self-correction during verification**.

The final Text Cipher data has **2,626 rows**. The **450 explicit recovery traces** teach an important behavior: after a PASS or FAIL decision, re-read the source word from the input query and verify again. If the second check disagrees with the first, trust the reread and correct the trace.

This is the most important Text Cipher contribution: the traces do not only teach how to decode; they teach how to recover from the model's own local verification mistakes. The recovery traces cover three cases:

1. **FAIL → PASS**: the first check wrongly rejects a correct candidate, then rereading fixes it.
2. **PASS → FAIL**: the first check wrongly accepts a bad candidate, then rereading catches the mistake and continues scanning.
3. **No candidate → reread**: the vocab scan finds no candidate, so the trace rereads the source word and scans again.

The row breakdown is:

| Text Cipher data | Rows |
|---|---:|
| Real traces | 1,576 |
| General synthetic / curriculum traces | 600 |
| Explicit recovery traces | 450 |
| **Total** | **2,626** |

### Case 1: FAIL becomes PASS after rereading

This teaches the model not to trust a first failed verification if the source word was copied incorrectly.

```text
rnuh -> near
letters r, n, u, h
r -> n agrees
n -> e agrees
u -> a conflicts
u -> e already exists
h -> r agrees
FAIL

re-read source word from input query
drpnh rnuh
rnuh
letters r, n, u, h
r -> n agrees
n -> e agrees
u -> a new
h -> r agrees
PASS confirm
add u -> a

choose near
```

Exact reference trace: [FAIL-to-PASS Text Cipher trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/text_cipher__text_cipher_reread_fail_to_pass__syn_tc_dp_phrase_alignment_0039.txt)

### Case 2: PASS becomes FAIL after rereading

This teaches the model not to stop after the first apparent PASS. If rereading contradicts the first check, the trace rejects that candidate and continues scanning.

```text
gub -> cat
letters g, u, b
g -> c new
u -> a agrees
b -> t new
PASS

re-read source word from input query
gub ahkkuex
gub
letters g, u, b
g -> c conflicts
v -> c already exists
u -> a agrees
b -> t new
FAIL
FAIL confirm, continue scanning

gub -> map
letters g, u, b
g -> m new
u -> a agrees
b -> p new
PASS

re-read source word from input query
gub ahkkuex
gub
letters g, u, b
g -> m new
u -> a agrees
b -> p new
PASS confirm
add g -> m
add b -> p

choose map
```

Exact reference trace: [PASS-to-FAIL Text Cipher trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/text_cipher__text_cipher_reread_pass_to_fail__syn_tc_dp_v2_extra_phrase_repeated_letter_failed_0032.txt)

### Case 3: no candidate means reread the source word

Sometimes the first decoded pattern produces zero vocabulary candidates. Instead of forcing a wrong answer, the trace teaches the model to reread the source word from the input query and scan again.

```text
Scan candidates for ?cve
(none)
no candidate
for no candidate, the source word was misread, re-read it

re-read source word from input query
lusdsk ljds
ljds
letters l, j, d, s
l -> c
j -> ? unknown
d -> v
s -> e
ljds -> c?ve
not fully mapped

Vocab pattern scan
same length and known letters match c?ve
...
cave -> c?ve match

Scan candidates for c?ve
cave
one candidate
for one candidate, verify then choose
```

Exact reference trace: [No-candidate recovery Text Cipher trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/text_cipher__text_cipher_no_candidate_recovery__syn_tc_case0_no_candidate_0038.txt)






## 1.4 Bit Manipulation

For Bit Manipulation, I want to specifically thank **Hui Kang** for publishing his traces. His Bit Manipulation traces were already in very good shape, and they became the backbone for this problem type in my final dataset.

My contribution here was intentionally modest: I mostly reused HuiKang's original traces, with minor modifications to fit the overall training format and trace style used in the rest of my corpus.

The final Bit Manipulation data has **6,117 rows**: 1602 real traces + 4515 synthetic traces. 

Reference trace: [Bit Manipulation trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/bit_manipulation__huikang_real_bit__b1f5a2e8.txt)






## 1.5 Numeric Equation

Numeric Equation Transformation Rules is the main data-engineering contribution in my solution. I aim to write traces that expose a compact and deterministic solver methodology.

The final Numeric Equation data has **4,530 rows**. The internal solver methodology covers about **89%** (651/732) of this problem type, so the synthetic rows were generated to teach that methodology, especially the rare branches that would otherwise appear too few times.

| Numeric Equation data | Rows |
|---|---:|
| Real traces | 651 |
| Targeted synthetic methodology traces | 3,879 |
| **Total** | **4,530** |

### 1.5.1 Solver vocabulary overview

Before describing the search flow, it helps to name the pieces of the solver. Every candidate rule combines an **input motif**, an **operation family**, and an **output format**.

| Component | Options used in the traces | Meaning |
|---|---|---|
| Input motif | `AB_CD`, `BA_DC` | Read operands as written (`AB`, `CD`) or reverse each operand (`BA`, `DC`). |
| Direct templates | `template0134`, `template3401` | Concatenate `AB` then `CD` (`ABCD`), or `CD` then `AB` (`CDAB`). |
| Addition families | `x+y`, `x+y-1`, `x+y+1` | Add the two interpreted operands, optionally with offset `-1` or `+1`. |
| Difference families | `x-y`, `y-x`, `abs(x-y)`, `min(x,y)-max(x,y)` | Try signed differences, absolute difference, and the negative min-minus-max branch. |
| Multiplication families | `x*y`, `x*y-1`, `x*y+1` | Multiply the two interpreted operands, optionally with offset `-1` or `+1`. |
| Modular families | `max(x,y)%min(x,y)`, `x%y`, `y%x` | Try remainder-style operations. |
| Output formats | `plain`, `rev`, `abs`, `abs_rev`, `op_prefix`, `op_suffix`, `op_prefix_if_neg`, `op_prefix_rev`, `rev_or_op_prefix_rev_if_neg`, `rev_or_op_suffix_if_neg`, `rev_or_op_suffix_rev_if_neg` | Render the numeric result directly, reversed, absolute-valued, or with the visible operator as prefix/suffix under specific sign branches. |

Here `x` and `y` mean the two operands after applying the input motif. Not every trace tries every option; the RHS-length rule below decides which operation families are worth testing, and the examples decide which output formats survive.

### 1.5.2 What makes this type hard

Each example has a simple surface form, `AB op CD = RHS`, but that same surface form can hide several different rules:

1. **Operand order**: use `AB_CD`, or reverse both operands and use `BA_DC`.
2. **Operation family**: try `x+y`, `x-y`, `y-x`, `x*y`, `x%y` with `+1/-1` offsets.
3. **Output rendering**: print the result as plain, reversed, absolute value, or with an operator prefix/suffix.
4. **Direct template**: sometimes there is no arithmetic; `template0134` means `ABCD`, and `template3401` means `CDAB`.

Trying all combinations would exceed the **7,680-token** generation budget. The trace therefore has to prune the search space while still showing every step faithfully.

### 1.5.3 A small trace DSL

Every trace starts from the same global solver order, and every candidate is named as `motif | operation | output format`. This naming convention gave the model a stable language for the search, and it made every trace auditable.

```text
We sequentially try direct templates, then motifs BA_DC and AB_CD. For each step, we choose the rule family from same-operator RHS length.

motif | operation | output format

BA_DC|x*y|common
AB_CD|x-y|common
BA_DC|x+y-1|common
```

### 1.5.4 RHS length as the pruning signal

The most useful observation was that the **length of the RHS** usually tells us which operation family is worth trying. This is the pruning rule that made the solver teachable. The first table gives the rule; the second table shows how often the routing branches appear.

| RHS length pattern | Candidate family to try | Exact families tried |
|---|---|---|
| length 4 | direct templates or multiplication | `template0134`, `template3401`, `x*y`, `x*y+1`, `x*y-1` |
| mixed length 3 and 4 | multiplication | `x*y`, `x*y+1`, `x*y-1` |
| length 3 only | addition or multiplication | `x+y`, `x+y-1`, `x+y+1`, `x*y`, `x*y+1`, `x*y-1` |
| mixed length 2 and 3 | addition | `x+y`, `x+y-1`, `x+y+1` |
| length 2 | addition, subtraction, or modular | `x+y`, `x+y-1`, `x+y+1`, `x-y`, `y-x`, `abs(x-y)`, `min(x,y)-max(x,y)`, `max(x,y)%min(x,y)`, `x%y`, `y%x` |
| mixed length 1 and 2 | subtraction or modular | `x-y`, `y-x`, `abs(x-y)`, `min(x,y)-max(x,y)`, `max(x,y)%min(x,y)`, `x%y`, `y%x` |
| length 1 | subtraction or modular | `x-y`, `y-x`, `abs(x-y)`, `min(x,y)-max(x,y)`, `max(x,y)%min(x,y)`, `x%y`, `y%x` |

| Routing branch | Traces containing it | Total occurrences |
|---|---:|---:|
| length 4 → direct templates or multiplication | 1,571 | 1,784 |
| length 3 → addition or multiplication | 1,060 | 1,202 |
| length 2 → addition, subtraction, or modular | 1,414 | 1,647 |
| length 1 → subtraction or modular | 515 | 539 |
| mixed length 3 and 4 → multiplication | 380 | 431 |
| mixed length 2 and 3 → addition | 441 | 495 |
| mixed length 1 and 2 → subtraction or modular | 452 | 503 |

This does not solve the problem by itself, but it prevents the trace from wasting tokens on irrelevant families. A typical trace fragment looks like this:

```text
Same operator RHS values are 8241
The RHS values have length 4, so use direct templates or multiplication
Try template0134,template3401,x*y,x*y+1,x*y-1
```

### 1.5.5 Same-operator examples first

The trace first compares the visible operators in the examples and the query. Examples with the same visible operator as the query are treated as the strongest evidence.

The normal flow is:

1. compare example operators,
2. collect same-operator examples,
3. infer the candidate family from same-operator RHS length,
4. try direct templates first,
5. if templates fail, try `BA_DC`, then `AB_CD`,
6. compute all output formats visibly,
7. keep only the common surviving formats,
8. apply the supported format to the query.

This ordering is important because it gives the model a deterministic path through the search, instead of asking it to guess from a large combinatorial space.

### 1.5.6 Motif verification for one-example cases

A risky case happens when the query operator has only **one** same-operator example. That single row may support a candidate branch, but it is weak evidence: the model can easily jump to that branch too early, even when another motif is the correct one.

To prevent this, the trace does **not** finalize after one supporting row. It first checks an additional visible operator group to verify whether the motif is consistent beyond the query operator.

This became one of the targeted synthetic-data branches. In the final corpus, this pattern appears in **1,109 traces**: **182 real traces** and **927 synthetic traces** generated to make the premature-branch-jump failure mode visible.

| Motif-verification behavior | Traces containing it |
|---|---:|
| `Only one same operator row supports this candidate` | 1,109 |
| `Verify motif BA_DC` | 1,109 |

Representative fragment:

```text
The format BA_DC|x*y|common supports the single same operator example
Only one same operator row supports this candidate, so do not finalize yet
Verify motif BA_DC using an additional helper operator group
```

### 1.5.7 Output formats and disagreement policies

After the raw arithmetic value is computed, the trace still has to decide how the answer is rendered. The output can be plain, reversed, absolute value, or include a visible operator as prefix/suffix.

The trace shows this as a visible table. Here the example is `82/15 = 8241`, and the candidate being tested is `BA_DC|x*y`:

```text
Example 82/15 = 8241
BA DC BA*DC rev plain op_prefix_if_neg rev_or_op_prefix_rev_if_neg op_prefix rev_or_op_suffix_rev_if_neg op_suffix op_prefix_rev abs_rev abs
28 51 1428 8241 1428 1428 8241 /1428 8241 1428/ /8241 8241 1428
Match
rev
rev_or_op_prefix_rev_if_neg
rev_or_op_suffix_rev_if_neg
abs_rev
```

Then the trace carries forward the shared surviving formats:

```text
Common
rev
rev_or_op_prefix_rev_if_neg
rev_or_op_suffix_rev_if_neg
abs_rev
```

Several branches need special handling beyond the normal "try candidate, keep common formats" flow. I made these branches explicit in the final corpus so the model could learn when to use each policy. The rare rows are especially important: without explicit policy traces, the model tends to guess a plausible rendering instead of applying the intended rule.

The purpose of the synthetic rows was to turn these low-count real patterns into repeated, auditable training patterns with the same decision flow.

| Branch or policy taught explicitly | Real traces | Synthetic traces | Total traces |
|---|---:|---:|---:|
| Direct-template success: `template0134` | 36 | 200 | 236 |
| Direct-template success: `template3401` | 36 | 200 | 236 |
| Operator-absence branch | 77 | 733 | 810 |
| Output disagreement resolved by voting | 18 | 59 | 77 |
| `BA_DC` with `x-y` negative rendering policy | 22 | 72 | 94 |
| `BA_DC` with `y-x` negative rendering policy | 2 | 20 | 22 |

For example, when common output formats disagree, the trace does not hide the choice. It counts votes and writes the winner explicitly:

```text
All common output formats do not agree
For motif BA_DC, use voting across common output formats
Output votes
6 has 2 votes
-6 has 2 votes
6- has 1 vote
Vote winner
6
```

### 1.5.8 Operator absence

Some queries use an operator symbol that never appears in the examples. This happened in **136** real Numeric Equation samples; the solver produced faithful traces for **77 / 136**, or **56.6%**, of them.

For these solved cases, the trace has two stages:

1. infer the motif and surviving output formats from all visible operator groups,
2. choose the absent query operator from a reverse-engineered symbol-to-operation candidate list.

That candidate list came from reverse engineering the real cases: for each visible symbol, I counted which operation families were most often needed to reach the gold answer, then used that frequency-ranked list as candidates. The trace then removes operation families already used by the visible examples and tests the remaining candidates by output agreement.

The final corpus contains **810 operator-absence rows**. A representative branch starts like this:

```text
same operator examples
none

For operator absence type, first infer the motif and output formats from all visible operator groups
Candidate output formats
rev
plain
op_prefix_if_neg
rev_or_op_prefix_rev_if_neg
rev_or_op_suffix_if_neg
rev_or_op_suffix_rev_if_neg
abs_rev
abs
```

After the visible groups fix the motif and common formats, the trace applies the reverse-engineered operator list. The policy is not just "always pick the first candidate": it first compares the remaining candidates by output agreement, and uses the first candidate only as the tie-breaker.

This made the operator-absence traces teach both parts of the method: infer the shared motif/output formats from available examples, then use the symbol-level statistics only for the missing query operator. In this exact trace, the absent query operator is `*`, whose candidate families are `x*y` and `x+y+1`:

```text
All visible operator groups support the following motif and common formats
Motif
BA_DC
Common
abs_rev

Use symbol mapping for query operator *
* -> x*y, x+y+1 PASS

Candidate operator families for *
x*y
x+y+1

Operator families used by the visible examples
x-y
x+y-1

Search remaining candidate operator families for *
Remaining candidate operator families
x*y
x+y+1

Check remaining candidate operator families by common-output agreement

Candidate x*y
Query
42*61
BA DC BA*DC abs_rev
24 16 384 483
Output votes
483 has 1 vote
Highest vote count
1

Candidate x+y+1
Query
42*61
BA DC BA+DC+1 abs_rev
24 16 41 14
Output votes
14 has 1 vote
Highest vote count
1

Choose operator candidate with highest output agreement vote count, if tie choose the first candidate
x*y
Apply format BA_DC|x*y|common to the query
```

### 1.5.9 Synthetic data design

The **3,879 synthetic Numeric Equation traces** were targeted at solver branches:

- direct template cases (`template0134` and `template3401`),
- normal multi-example `BA_DC` and `AB_CD` branches,
- one same-operator example with motif verification,
- operator-absence cases,
- output-format rendering branches, including common-format intersection and voting,
- negative `x-y` / `y-x` rendering policies,
- rare RHS-length and operation-family combinations,
- rare answer-rendering forms, such as operator-prefixed/suffixed outputs and `{` / `}` answers.

The goal was not to make every rare policy dominate the dataset. The goal was to generate enough examples for each policy to become visible to the model, while keeping its count proportional enough that it did not overpower the common real-trace patterns.

### 1.5.10 Exact reference traces

- [Direct-template Numeric Equation trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/numeric_equation_transformation_rules__real__047c4111.txt)
- [Motif-verification Numeric Equation trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/numeric_equation_transformation_rules__real__01cd504a__rich.txt)
- [Operator-absence Numeric Equation trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/numeric_equation_transformation_rules__real__1eaf6228__rich.txt)
- [Output-disagreement / voting Numeric Equation trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/numeric_equation_transformation_rules__real__04171e29__rich.txt)
- [Negative-policy Numeric Equation trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/numeric_equation_transformation_rules__real__10fdae00__rich.txt)





## 1.6 Cryptarithm / Symbol Transform

This problem type was one of the reasons I was attracted to the competition: I believed I could write good traces for these symbolic equation puzzles. The observation is natural for a cryptarithm-style problem: each symbol represents an integer from **0 to 9**, and two distinct symbols have different values. Once the motif (`AB_CD` or `BA_DC`) and operator family (`+`, `-`, `*`, etc.) are known, the problem becomes a system of digit equations.

I wrote archived deep-solver traces for the common encrypted-digit families, especially `BA_DC|x*y|rev` and `AB_CD|x+y|plain`. The scratch work contains **over 200** such traces: **162** BA/DC-reversal full traces and **49** AB/CD addition/multiplication golden traces. I spent roughly **3 weeks** of the competition on this type alone.
The regret is that this was not enough. After training, it became clear that these algebraic traces probably needed at least a **10x synthetic expansion** before the model could learn them reliably. With a full-time job and only weekends to work on the competition, I decided to stop investing here and focus on the types with higher return.

So the final Symbol Transform data is intentionally conservative. It remained the weakest type in my final system, with about **8.7%** solve rate. The final training data used only three simple branches. Here, real means train-origin rows from `train.csv`.

| Final Symbol Transform branch | Real traces | Synthetic traces | Total traces |
|---|---:|---:|---:|
| Direct-template traces | 59 | 541 | 600 |
| Operator-absence direct-template / guess heuristic | 164 | 0 | 164 |
| Non-template unresolved cases with explicit guess fallback | 600 | 0 | 600 |
| **Total** | **823** | **541** | **1,364** |

- **Direct template:** if same-operator examples support `template0134` (`ABCD`) or `template3401` (`CDAB`), apply the passing template to the query. Reference trace: [Direct-template Symbol Transform trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/symbol_transform__real__9fc69c17.txt)
- **Operator absence:** if the query operator never appears in the examples, use a simple `template0134` heuristic. Reference trace: [Operator-absence Symbol Transform trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/symbol_transform__op_ab_guess_0134_correct__258b796b.txt)
- **Unresolved cases:** if direct templates fail and the query operator appears in the examples, explicitly mark that no reliable rule was identified and train a guess fallback. Reference trace: [Unresolved Symbol Transform guess trace](https://github.com/dtdo90/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/open_contribution_awards/representative_traces/v5_exact/symbol_transform__symbol_transform_unreliable_pattern_guess__6c7f24b7.txt)




# 2. Fine-Tuning

Once the trace data stabilized, the fine-tuning recipe had two practical pieces:

1. **Selective token weighting**, so important decision tokens receive more gradient,
2. **Category-balanced accumulation**, so each gradient-accumulation window contains a mix of problem types.

This subsection covers token weighting. Balanced accumulation is covered next.

## 2.1 Selective Token Weighting

The goal was not to upweight an entire problem type. The goal was to upweight the tokens where the model is most likely to choose a wrong branch, apply a rare policy incorrectly, or make a bad verification decision.

Most tokens stay at weight **1.0**. With `--decision-weight 2`, selected decision spans receive weight **2.0**. Numeric Equation also has a small critical tier for answer-critical policy outputs; with `--decision-weight 2`, that tier becomes **3.0**.

### 2.1.1 Weighted loss

The trainer stores a `label_weights` vector aligned with `input_ids`. Prompt tokens are already masked with `labels = -100`, so their weights are `0.0`; normal assistant tokens get `1.0`; selected decision tokens get higher weight.

The denominator is the number of labeled tokens, not the sum of weights. This makes weighting act as **selective gradient amplification** instead of renormalizing the example back down.

The implementation is weighted cross entropy:


In [ ]:
def weighted_ce_loss(shift_logits, shift_labels, shift_weights, denom):
    """Weighted CE used when label_weights are present."""
    import torch.nn.functional as F

    vocab = shift_logits.size(-1)
    per_token = F.cross_entropy(
        shift_logits.reshape(-1, vocab).float(),
        shift_labels.reshape(-1),
        ignore_index=-100,
        reduction="none",
    )

    keep = shift_labels.reshape(-1) != -100
    weights = shift_weights.reshape(-1).to(per_token.dtype)
    numerator = (per_token * weights)[keep].sum()
    return numerator / denom.clamp_min(1.0)


### 2.1.2 How weights are attached

The weighter first marks important character spans, then maps those character weights back to tokenizer offsets. This avoids fragile assumptions about token boundaries.

A simplified Text Cipher-style `build_char_weights` looks like this:


In [ ]:
import re


def build_char_weights(text: str, *, high: float = 2.0, base: float = 1.0) -> list[float]:
    """Toy example: mark important completion characters before tokenization."""
    weights = [base] * len(text)

    lines: list[tuple[int, str]] = []
    offset = 0
    for line in text.split("\n"):
        lines.append((offset, line))
        offset += len(line) + 1

    def mark(start: int, end: int, weight: float = high) -> None:
        for k in range(max(start, 0), min(end, len(weights))):
            weights[k] = max(weights[k], weight)

    for start, line in lines:
        stripped = line.strip()
        lead = len(line) - len(line.lstrip())
        body_start = start + lead
        body_end = start + len(line.rstrip())

        if stripped in {"PASS", "FAIL", "PASS confirm", "FAIL confirm, continue scanning"}:
            mark(body_start, body_end)
            continue

        if stripped.startswith("add ") or stripped.startswith("choose "):
            mark(body_start, body_end)
            continue

        # Promote only the decision word in vocab scan rows, not every "no" row.
        match = re.search(r"\bmatch$", stripped)
        if match:
            mark(body_start + match.start(), body_start + match.end())

    return weights


The production weighters contain more problem-specific span rules, but the core idea is the same: mark only the characters that carry the decision. The wrapper below then copies those completion-side character weights into the full prompt-plus-completion sequence and converts them to token weights.


In [ ]:
def completion_label_weights(tokenizer, prompt_text, completion_text, *, high=2.0, base=1.0):
    """Return one weight per token in prompt_text + completion_text."""
    full_text = prompt_text + completion_text
    enc = tokenizer(full_text, add_special_tokens=False, return_offsets_mapping=True)
    prompt_token_count = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])

    char_weights = [base] * len(full_text)
    completion_weights = build_char_weights(completion_text, high=high, base=base)
    prompt_chars = len(prompt_text)
    for i, weight in enumerate(completion_weights):
        char_weights[prompt_chars + i] = weight

    return token_weights_from_offsets(
        enc["offset_mapping"],
        char_weights,
        prompt_token_count=prompt_token_count,
        base=base,
    )

In the final recipe, token weighting was applied only to **Text Cipher**, **Numeric Equation**, and **Symbol Transform**; the other problem types stayed at uniform weight **1.0**.

For diagnostic ablations, setting `--decision-weight 1` reduces everything to ordinary uniform-token training. The final trainer dispatches by problem type. Each hard trace type has its own weighter because the useful decision spans are different:


In [ ]:
CATEGORY_WEIGHTERS = {
    "Text Cipher": "text_cipher_loss_weights.completion_label_weights",
    "Symbol Transform": "symbol_transform_loss_weights.completion_label_weights",
    "Numeric Equation Transformation Rules": "numeric_equation_loss_weights.completion_label_weights",
}

### 2.1.3 What gets weighted

The weighting rules differ by problem type because the failure modes differ. One global rule is that the appended boxed answer after `</think>` stays weight **1.0**. It is a formatting echo, not the reasoning decision.

| Problem type | Higher-weight spans | Kept at weight 1 |
|---|---|---|
| Text Cipher | source-letter anchors, summary mappings, candidate words, `match`, `PASS`/`FAIL`, confirmation, `add ...`, decoded phrase | vocabulary dump, ordinary `no` rows, repeated prose, post-`</think>` boxed echo |
| Numeric Equation | RHS-routing payloads, supported formats, common surviving formats, helper verification, query output rows, vote counts/winners, rare negative policies | repeated table scaffold, failed candidate rows, `Try BA_DC first`, `The current format is ...`, `Match`, `Common` |
| Symbol Transform | operator comparison verdicts, direct-template AB/CD parse values, `Match`/`No match`, produced value after `gives`, final in-think boxed answer | echoed examples, variable naming, generic routing prose, post-`</think>` boxed echo |

### 2.1.4 Example: Text Cipher

Here `[[...]]` means weight 2. Text Cipher failures often came from skipped repeated letters or early PASS/FAIL decisions, so the weighted spans focus on letters, mappings, candidate verification, confirmation, and the chosen word.

```text
Summary character mappings
[[u -> q]]
[[z -> u]]
[[v -> e]]
[[d -> n]]

uzvvd
letters [[u, z, v, v, d]]
u -> [[? unknown]]
z -> [[u]]
v -> [[e]]
v -> [[e]]
d -> [[n]]
uzvvd -> [[?ueen]]
not fully mapped

Vocab pattern scan
same length and known letters match [[?ueen]]
queen -> ?ueen [[match]]

Scan candidates for ?ueen
[[queen]]
one candidate
for one candidate, verify then choose

[[uzvvd -> queen]]
letters [[u, z, v, v, d]]
u -> [[q new]]
z -> [[u agrees]]
v -> [[e agrees]]
v -> [[e agrees]]
d -> [[n agrees]]
[[PASS]]

re-read source word from input query
uzvvd vjnaclvo etckv bclvor
[[uzvvd]]
letters [[u, z, v, v, d]]
u -> [[q new]]
z -> [[u agrees]]
v -> [[e agrees]]
v -> [[e agrees]]
d -> [[n agrees]]
[[PASS confirm]]
[[add u -> q]]

choose [[queen]]
```

### 2.1.5 Example: Numeric Equation

Numeric Equation failures often happen at branch choice, output rendering, or voting. In the example below, `[[...]]` is weight 2 and `[[[...]]]` is the rarer critical tier that carries weight 3.

```text
Same operator RHS values are 8241
The RHS values have [[length 4]], so use [[direct templates or multiplication]]
Try template0134,template3401,x*y,x*y+1,x*y-1

The format [[BA_DC|x*y|common]] supports the single same operator example
Only one same operator row supports this candidate, so do not finalize yet
Verify motif BA_DC using an additional helper operator group

Common
[[rev]]
[[rev_or_op_prefix_rev_if_neg]]
[[rev_or_op_suffix_rev_if_neg]]
[[abs_rev]]

Apply format [[BA_DC|x*y|common]] to the query

Query
42*61
BA DC BA*DC abs_rev
[[[24 16 384 483]]]
All common output formats agree on [[[483]]]
```

For voting cases, the vote rows and winner are also critical:

```text
All common output formats do not agree
For motif BA_DC, use voting across common output formats
Output votes
[[[6 has 2 votes]]]
[[[-6 has 2 votes]]]
[[6- has 1 vote]]
Vote winner
[[[6]]]
```

### 2.1.6 Example: Symbol Transform

For direct-template Symbol Transform rows, the important part is the concrete parse and comparison. The example below keeps weighting selective: it reinforces tokens that choose or verify a branch, while ordinary repeated trace scaffolding remains weight 1.

```text
Query [[#'*:@]]
Query operator is [[*]]

Compare example operators
/$-/[ = $ operator [[-]]
[[different]]
|@*:] = |@:] operator [[*]]
[[same]]

Try template0134
template0134 means AB followed by CD

Example |@*:] = |@:]
AB = [[|@]]
operator = [[*]]
CD = [[:]]
[[|@ followed by :] gives |@:] vs |@:]]
[[Match]]

template0134 passes all examples
For direct templates, apply the passing template to get the answer

Apply template0134 to the query
Query
#'*:@
AB = [[#']]
operator = [[*]]
CD = [[:@]]
[[#' followed by :@ gives #':@]]

Answer: [[\boxed{#':@}]]
```


## 2.2 Category-Balanced Accumulation

The second fine-tuning trick was balancing each gradient-accumulation window by problem category.

This mattered because the final corpus was very uneven. With per-device batch size **1** and gradient accumulation **8**, each optimizer step sees only eight sequences. If those eight sequences are sampled naively, several consecutive optimizer steps can be dominated by the largest categories, especially Bit Manipulation and Numeric Equation. That makes smaller categories appear in bursts instead of giving the optimizer a steady mixed signal.

The final 19,404-row corpus had this category distribution:

| Problem type | Rows |
|---|---:|
| Bit Manipulation | 6,117 |
| Numeric Equation Transformation Rules | 4,530 |
| Text Cipher | 2,626 |
| Gravity | 1,597 |
| Unit Conversion | 1,594 |
| Numeral System | 1,576 |
| Symbol Transform | 1,364 |
| **Total** | **19,404** |

Balanced accumulation does **not** change the data, loss, or prompt format. It only changes the training order so that each accumulation window is more evenly spread across categories.


In [ ]:
import math
import random


def balanced_accumulation_order(groups: list[str], *, effective_batch_size: int, seed: int) -> list[int]:
    """Spread category labels across optimizer-step accumulation windows."""
    if effective_batch_size <= 1:
        return list(range(len(groups)))

    rng = random.Random(seed)
    n_windows = math.ceil(len(groups) / effective_batch_size)
    windows: list[list[int]] = [[] for _ in range(n_windows)]
    window_order = list(range(n_windows))
    rng.shuffle(window_order)

    by_group: dict[str, list[int]] = {}
    for index, group in enumerate(groups):
        by_group.setdefault(group or "unknown", []).append(index)

    assigned = 0
    for group in sorted(by_group):
        indices = by_group[group]
        rng.shuffle(indices)
        for index in indices:
            windows[window_order[assigned % n_windows]].append(index)
            assigned += 1

    order: list[int] = []
    for window in windows:
        rng.shuffle(window)
        order.extend(window)
    return order


The important design choice is that the sampler balances at the **optimizer-step window** level, not at the individual microbatch level. In my setup, `effective_batch_size = 1 * 8 = 8`, so the sampler tries to spread categories across each group of eight microbatches before the optimizer update.

The trainer activates this sampler only when `--balanced-accumulation` is passed:


In [ ]:
trainer_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "data_collator": MaskedCausalLMDataCollator(tokenizer),
    "args": trainer_args,
}

if args.balanced_accumulation:
    trainer_kwargs.update(
        {
            "balanced_accumulation_groups": [example.category for example in train_examples],
            "balanced_accumulation_effective_batch_size": (
                args.per_device_train_batch_size * args.gradient_accumulation_steps
            ),
            "balanced_accumulation_seed": 42,
        }
    )


### 2.2.1 Final training recipe

The final run used two epochs in spirit, implemented as two scripts because of the **24-hour Google Colab runtime limit**. The first epoch already took about **21 hours**, so I saved the adapter and launched the continuation as a separate run.

**Epoch 1** trained on the full **19,404-row** corpus with selective token weighting and balanced accumulation.

**Epoch 2** continued from the first adapter and trained for one more epoch on **13,298 rows**: all train-origin rows plus the **3,879 synthetic Numeric Equation** rows. I kept Numeric Equation synthetic rows in the second pass because this was the most methodology-heavy type, and removing its synthetic solver traces made the method weaker.

The commands below are the exact recipe I would use to reproduce the final training setup. Batch size 1, gradient accumulation 8, and balanced accumulation are defaults in the scripts, so they are not repeated in the command.

```bash
python3 train_sft_single_phase.py --decision-weight 2
```

and

```bash
python3 train_sft_single_phase_last_dance.py --adapter-dir outputs/sft_single_phase/adapter --decision-weight 2 
```